# Scaled-up, night/dark-robust joint training (road + building + point)

**Cloud GPU only -- do not run locally.** This notebook is the moderate-scale, multi-epoch
successor to `train_unet_joint.ipynb` (which is a frozen 2-epoch/~150-tile CPU proof of concept).
Its goal is specifically **robustness to dark and night satellite imagery**, which the proof-of-
concept model was never exposed to and fails on.

Two independent lines of defense against dark/night input, combined (per the project's night-
robustness plan):

1. **Real illumination-invariant data**: SpaceNet 6 (`datasets/spacenet6.py`) -- Capella Space SAR
   imagery over Rotterdam. SAR is an active sensor (it illuminates its own target with radar), so
   it is genuinely illumination-invariant -- it looks the same day or night. Building-footprint
   labels only (SN6 has no road/point annotations), folded in as a third `SOURCE_CLASSES` entry
   with zero architecture change (converted to pseudo-RGB at dataset-build time, see
   `datasets/spacenet6.py::sar_bands_to_pseudo_rgb`).
2. **Synthetic low-light augmentation** (`datasets/augment.py::simulate_low_light`) applied to a
   fraction of training patches cut from the daytime-labeled SpaceNet/Potsdam sources -- gamma
   darkening, brightness/contrast/saturation jitter, CLAHE, and sensor-noise injection, since no
   public dataset exists with real night-labeled optical satellite imagery paired with full
   road+building+point semantic labels (verified during planning).

A dedicated **dark-test evaluation** section near the end applies the same low-light simulation at
fixed severities to the real held-out test tiles and reports `evaluate_all()` metrics side by side
across severities -- this is the concrete evidence for whether the model actually got more robust,
not just that dark examples were included in training.

**Scale** (moderate, per project scope decision): ~1,000-1,500 base tiles across 3 sources
(SpaceNet SN2/SN3 x4 cities, Potsdam, SpaceNet6), patchified into many thousands of 256x256
patches, ~40 epochs, mixed precision, on a single CUDA GPU -- should run in hours, not require
multi-GPU. Never modifies the frozen `train_unet.ipynb`/`train_unet_joint.ipynb` baselines; new
checkpoints land at `models/unet_joint_4class_scaled*.pt`.


## Cloud GPU setup (mandatory -- this notebook has no CPU fallback)

Unlike the frozen CPU baselines, this notebook is GPU-only by design: a full run at this scale on
CPU would take far too long to be practical. The cell below clones the repo if it isn't already
present, installs all dependencies (including a CUDA-enabled `torch` build -- the rest of the repo
documents a CPU-wheel install, since the baselines are CPU-only; this notebook explicitly needs the
GPU wheel instead), and hard-fails if no CUDA device is visible rather than silently falling back to
CPU and taking hours/days longer than expected.


In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/DanielZGeorge/PLEM.git"
REPO_DIR = "/content/PLEM" if "google.colab" in sys.modules else os.path.abspath("..")

code_present = (os.path.isdir(os.path.join(REPO_DIR, "datasets"))
                and os.path.isdir(os.path.join(REPO_DIR, "models")))

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
elif code_present:
    # Uploaded directly (e.g. a zipped copy of the repo pushed to a cloud GPU box) rather
    # than git-cloned -- the code is already here, so skip cloning/pulling entirely instead
    # of failing on a non-empty destination.
    print(f"Using manually uploaded copy at {REPO_DIR} (no .git dir found, skipping clone/pull).")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                os.path.join(REPO_DIR, "requirements.txt")], check=True)

# CUDA-enabled torch build -- replaces the CPU-wheel instruction the rest of
# the repo's CLAUDE.md documents, since this notebook is GPU-only by design.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tqdm"], check=True)

os.chdir(os.path.join(REPO_DIR, "notebooks"))
print(f"Repo ready at {REPO_DIR}, cwd set to {os.getcwd()}")


Using manually uploaded copy at /home1/dzgeorge/PLEM_cloud_upload (no .git dir found, skipping clone/pull).


Repo ready at /home1/dzgeorge/PLEM_cloud_upload, cwd set to /home1/dzgeorge/PLEM_cloud_upload/notebooks


In [2]:
import sys, os, random
from pathlib import Path
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader

assert torch.cuda.is_available(), (
    "No CUDA device visible -- this notebook is GPU-only by design (unlike the frozen CPU "
    "baselines). Run it on a cloud GPU server, not locally."
)

from metrics import evaluate_all
from models.unet import SmallUNet
from losses.multitask import PLEMMultiTaskLoss
from datasets.joint import load_joint_tiles, class_mask_for_source, SOURCE_CLASSES, NUM_CLASSES
from datasets.spacenet import build_spacenet_sample
from datasets.potsdam import load_kaggle_dataset, build_potsdam_sample
from datasets.spacenet6 import build_spacenet6_sample
from datasets.augment import simulate_low_light

DATA_DIR = Path("..").resolve() / "data"
MODELS_DIR = Path("..").resolve() / "models"
MODELS_DIR.mkdir(exist_ok=True)

SEED = 0
PATCH = 256
DEVICE = torch.device("cuda")
print(f"Using device: {DEVICE} ({torch.cuda.get_device_name(0)})")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

CLASS_NAMES = {0: "background", 1: "road", 2: "building", 3: "point"}


def overlay4(label):
    """Color-code a 4-class label map: road red, building green, point yellow."""
    o = np.zeros((*label.shape, 3), dtype=np.uint8)
    o[label == 1] = [255, 0, 0]
    o[label == 2] = [0, 255, 0]
    o[label == 3] = [255, 255, 0]
    return o


Using device: cuda (NVIDIA A100 80GB PCIe)


## Build the SpaceNet + Potsdam + SpaceNet6 caches (moderate scale)

Unconditional (not Colab-gated) -- a fresh cloud box always starts with an empty `data/` cache, so
this always needs to run once. Skips a source entirely if its cache directory already has files
(so re-running this cell after an interrupted run is cheap). Scale targets, within the project's
approved "moderate" band:

- **SpaceNet** (`datasets/spacenet.py`): 4 cities x `n_tiles=175` (up from the proof-of-concept's
  25/city) -- road+building, no point class.
- **Potsdam** (`datasets/potsdam.py`): `n_crops=200` is a non-binding cap -- the Kaggle mirror's
  actual available crop count is the real limiter (the full Potsdam sample is much smaller than
  200; this just avoids artificially capping below what's actually available). Building+point, no
  road class. Needs a one-time, user-side Kaggle API token (`~/.kaggle/kaggle.json`) -- if missing,
  this cell prints instructions and continues with SpaceNet+SpaceNet6-only data rather than
  crashing.
- **SpaceNet6** (`datasets/spacenet6.py`, new): `n_tiles=300` -- real SAR imagery, building only,
  the illumination-invariant half of the night-robustness approach. Downloads one large tarball
  (multi-GB, one-time) rather than many small per-tile objects.


In [3]:
CITIES = ["Vegas", "Khartoum", "Paris", "Shanghai"]


def _count_source_npz(name):
    """Cached .npz count using the SAME globbing load_joint_tiles uses, so this
    matches what actually loads. A recursive rglob would over-count a cache
    written to the wrong depth and hide a layout bug."""
    d = DATA_DIR / name
    if not d.is_dir():
        return 0
    if name == "spacenet":
        n = sum(1 for cd in d.iterdir() if cd.is_dir() for _ in cd.glob("*.npz"))
        return n or len(list(d.rglob("*.npz")))  # rglob fallback mirrors load_joint_tiles
    return len(list(d.glob("*.npz")))


if not os.path.isdir(DATA_DIR / "spacenet") or not any((DATA_DIR / "spacenet").iterdir()):
    print("data/spacenet is empty -- building the scaled SpaceNet sample (SN2+SN3)...")
    for city in CITIES:
        # cache_dir WITHOUT the city: build_spacenet_sample appends <city>/ itself
        # (datasets/spacenet.py), and load_joint_tiles expects data/spacenet/<city>/*.npz.
        # Passing DATA_DIR/"spacenet"/city here produced data/spacenet/<city>/<city>/*.npz
        # and load_joint_tiles found zero SpaceNet tiles.
        build_spacenet_sample(city, n_tiles=175, n_road_candidates=400,
                               cache_dir=str(DATA_DIR / "spacenet"), seed=0)
else:
    print("data/spacenet already populated -- skipping.")

# Fail fast BEFORE the multi-GB SpaceNet6 download if SpaceNet yielded nothing.
assert _count_source_npz("spacenet") > 0, (
    "0 SpaceNet tiles cached -- the build failed or the cache landed at an "
    "unexpected path. Fix this before continuing; the SN6 download below is expensive."
)

if not os.path.isdir(DATA_DIR / "potsdam") or not any((DATA_DIR / "potsdam").iterdir()):
    try:
        load_kaggle_dataset(dest=str(DATA_DIR / "potsdam_raw"))
        build_potsdam_sample(
            n_crops=200, seed=0, cache_dir=str(DATA_DIR / "potsdam"),
            raw_dir=str(DATA_DIR / "potsdam_raw"), extract_buildings=True,
        )
    except Exception as e:
        print(f"Could not build the Potsdam cache automatically ({e}). Set up "
              f"~/.kaggle/kaggle.json (see potsdam_data_prep.ipynb) and re-run this cell.")
else:
    print("data/potsdam already populated -- skipping.")

# Fail fast here too -- cell 7's `assert len(by_source) == 3` requires Potsdam, and
# catching a zero-yield now (rather than after the 42 GB SN6 pull) saves an hour.
assert _count_source_npz("potsdam") > 0, (
    "0 Potsdam tiles cached -- the point class can't be trained/evaluated and the "
    "3-source assert later will fail. Set up ~/.kaggle/kaggle.json and re-run this cell."
)

if not os.path.isdir(DATA_DIR / "spacenet6") or not any((DATA_DIR / "spacenet6").iterdir()):
    print("data/spacenet6 is empty -- building the SpaceNet6 SAR sample (this downloads a "
          "multi-GB tarball once)...")
    build_spacenet6_sample(
        n_tiles=300, cache_dir=str(DATA_DIR / "spacenet6"), seed=0,
        # explicit tarball_path -- the function's own default is a relative string
        # that would resolve against notebooks/ (cell 2 chdir'd there). Anchor it
        # to the top-level data/ dir like every other cache path here.
        tarball_path=str(DATA_DIR / "spacenet6_raw" / "SN6_buildings_AOI_11_Rotterdam_train.tar.gz"),
    )
else:
    print("data/spacenet6 already populated -- skipping.")

for name in ("spacenet", "potsdam", "spacenet6"):
    print(f"  {name}: {_count_source_npz(name)} cached tiles")


data/spacenet already populated -- skipping.
data/potsdam already populated -- skipping.
data/spacenet6 already populated -- skipping.
  spacenet: 550 cached tiles
  potsdam: 200 cached tiles
  spacenet6: 283 cached tiles


## Load tiles and split train/val/test (per source, then concatenate)

`load_joint_tiles()` now loads all **three** sources, tagging each with its `source` and
`class_mask` (`datasets/joint.py::SOURCE_CLASSES` -- `spacenet6` added as a third entry,
building-only). Each source is split 70/15/15 **independently** at the tile level before
concatenating, same rationale as the proof-of-concept notebook: a single pooled-then-permuted split
risks an unlucky seed starving the smallest source from a split entirely.


In [4]:
def split_tiles(tiles, seed=SEED):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(tiles))
    n = len(tiles)
    n_train = int(round(0.70 * n))
    n_val = int(round(0.15 * n))
    train = [tiles[i] for i in perm[:n_train]]
    val = [tiles[i] for i in perm[n_train:n_train + n_val]]
    test = [tiles[i] for i in perm[n_train + n_val:]]
    return train, val, test


all_tiles = load_joint_tiles(
    spacenet_dir=DATA_DIR / "spacenet", potsdam_dir=DATA_DIR / "potsdam",
    spacenet6_dir=DATA_DIR / "spacenet6",
)
by_source = {}
for t in all_tiles:
    by_source.setdefault(t["source"], []).append(t)
print(f"Loaded {len(all_tiles)} tiles: " + ", ".join(f"{k}={len(v)}" for k, v in by_source.items()))

train_tiles, val_tiles, test_tiles = [], [], []
for source, tiles in by_source.items():
    tr, va, te = split_tiles(tiles, seed=SEED)
    train_tiles += tr
    val_tiles += va
    test_tiles += te
    print(f"  {source}: train={len(tr)} val={len(va)} test={len(te)}")

print(f"total: train={len(train_tiles)}  val={len(val_tiles)}  test={len(test_tiles)}")
assert len(by_source) == 3, (
    f"Expected all 3 sources (spacenet, potsdam, spacenet6) to be present, got {list(by_source)} "
    f"-- check the dataset-build cell above for a source that failed silently."
)


Loaded 1033 tiles: spacenet=550, potsdam=200, spacenet6=283
  spacenet: train=385 val=82 test=83
  potsdam: train=140 val=30 test=30
  spacenet6: train=198 val=42 test=43
total: train=723  val=154  test=156


## Patchify

Identical to `train_unet_joint.ipynb`'s `pad_to_multiple`/`patchify_tile`/`predict_tile` -- these
are already source-agnostic (a patch just carries its source tile's fixed `class_mask` through
unchanged), so a third source needs no changes here.


In [5]:
def pad_to_multiple(image, label, patch=PATCH):
    h, w = label.shape
    new_h = int(np.ceil(h / patch)) * patch
    new_w = int(np.ceil(w / patch)) * patch
    pad_h, pad_w = new_h - h, new_w - w
    image_p = np.pad(image, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
    label_p = np.pad(label, ((0, pad_h), (0, pad_w)), mode="constant", constant_values=0)
    return image_p, label_p, (h, w)


def patchify_tile(image, label, class_mask, patch=PATCH):
    """Returns a list of (image_patch, label_patch, class_mask) covering the whole tile."""
    image_p, label_p, _ = pad_to_multiple(image, label, patch)
    ph, pw = label_p.shape
    patches = []
    for r in range(0, ph, patch):
        for c in range(0, pw, patch):
            patches.append((
                image_p[r:r + patch, c:c + patch], label_p[r:r + patch, c:c + patch], class_mask,
            ))
    return patches


def predict_tile(model, image, orig_label_shape, patch=PATCH, device=DEVICE):
    """Runs the model patch-by-patch over a full tile and stitches an HxW prediction map."""
    dummy_label = np.zeros(orig_label_shape, dtype=np.uint8)
    image_p, _, orig_shape = pad_to_multiple(image, dummy_label, patch)
    ph, pw = image_p.shape[:2]
    canvas = np.zeros((ph, pw), dtype=np.uint8)
    model.eval()
    with torch.no_grad():
        for r in range(0, ph, patch):
            for c in range(0, pw, patch):
                patch_img = image_p[r:r + patch, c:c + patch]
                x = torch.from_numpy(patch_img.transpose(2, 0, 1).astype(np.float32) / 255.0)
                x = x.unsqueeze(0).to(device)
                logits = model(x)
                pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
                canvas[r:r + patch, c:c + patch] = pred
    h, w = orig_shape
    return canvas[:h, :w]


# Point-bearing patches are rare: only Potsdam annotates the point class, and
# most Potsdam crops (kept for their building coverage under
# extract_buildings=True/min_building_pixels=1) contain no point instances at
# all. Left un-oversampled they're a fraction of a percent of the ~17k training
# patches, far too sparse for the point channel to learn against a 4-term loss.
# Replicate each point-bearing patch POINT_OVERSAMPLE times; the per-__getitem__
# random flip + dark augmentation mean the copies are not identical at train time.
POINT_OVERSAMPLE = 6


def expand_with_point_oversampling(patches, factor=POINT_OVERSAMPLE):
    out, n_point = [], 0
    for img, lab, cm in patches:
        out.append((img, lab, cm))
        if (lab == 3).any():
            n_point += 1
            out.extend([(img, lab, cm)] * (factor - 1))
    return out, n_point


raw_train_patches = [p for t in train_tiles for p in patchify_tile(t["image"], t["label"], t["class_mask"])]
train_patches, n_point_patches = expand_with_point_oversampling(raw_train_patches)
val_patches = [p for t in val_tiles for p in patchify_tile(t["image"], t["label"], t["class_mask"])]
print(f"train_patches={len(train_patches)}  ({len(raw_train_patches)} raw; "
      f"{n_point_patches} point-bearing patches replicated x{POINT_OVERSAMPLE})")
print(f"val_patches={len(val_patches)}  (patch={PATCH}px)")


train_patches=17473  (17168 raw; 61 point-bearing patches replicated x6)
val_patches=3654  (patch=256px)


## Dataset / DataLoader with dark-augmentation

`PatchDataset` gains a `dark_aug_prob` parameter: on each `__getitem__` call (train split only),
with probability `dark_aug_prob` the image is darkened via `simulate_low_light` (randomized
severity, drawn fresh per call) *before* the existing flip augmentation -- the label is never
touched, since severity is purely a pixel-value transform. A per-batch source-mix sanity check
(distinguishing all 3 sources by their distinct `class_mask` patterns:
spacenet=`[1,1,1,0]`, potsdam=`[1,0,1,1]`, spacenet6=`[1,0,1,0]`) confirms all 3 sources actually
appear across training batches.


In [6]:
class PatchDataset(Dataset):
    def __init__(self, patches, augment=False, dark_aug_prob=0.0):
        self.patches = patches
        self.augment = augment
        self.dark_aug_prob = dark_aug_prob
        self._rng = np.random.default_rng(SEED)

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        image, label, class_mask = self.patches[idx]
        if self.dark_aug_prob > 0 and random.random() < self.dark_aug_prob:
            image = simulate_low_light(image, self._rng)
        if self.augment:
            if random.random() < 0.5:
                image, label = image[:, ::-1], label[:, ::-1]
            if random.random() < 0.5:
                image, label = image[::-1, :], label[::-1, :]
        image_t = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1)).astype(np.float32) / 255.0)
        label_t = torch.from_numpy(np.ascontiguousarray(label).astype(np.int64))
        class_mask_t = torch.from_numpy(np.ascontiguousarray(class_mask).astype(np.float32))
        return image_t, label_t, class_mask_t


BATCH_SIZE = 48
DARK_AUG_PROB = 0.4

train_loader = DataLoader(
    PatchDataset(train_patches, augment=True, dark_aug_prob=DARK_AUG_PROB),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    PatchDataset(val_patches, augment=False, dark_aug_prob=0.0),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
)

# Per-batch source-mix sanity check: each source has a distinct class_mask
# pattern (see markdown above), so class_mask alone identifies the source.
# islice, not list(...)[:3] -- the latter materializes a full training epoch
# (every batch + a full pass of dark augmentation) just to print 3 rows.
import itertools
first_batches = list(itertools.islice(train_loader, 3))
for i, (_, _, cm) in enumerate(first_batches):
    n_spacenet = int(((cm[:, 1] == 1) & (cm[:, 3] == 0)).sum())
    n_sn6 = int(((cm[:, 1] == 0) & (cm[:, 3] == 0)).sum())
    n_potsdam = int((cm[:, 1] == 0).sum()) - n_sn6
    print(f"batch {i}: spacenet={n_spacenet}  potsdam={n_potsdam}  spacenet6={n_sn6}  "
          f"(batch_size={cm.shape[0]})")


batch 0: spacenet=41  potsdam=0  spacenet6=7  (batch_size=48)
batch 1: spacenet=40  potsdam=3  spacenet6=5  (batch_size=48)
batch 2: spacenet=40  potsdam=2  spacenet6=6  (batch_size=48)


## 4-class U-Net + `PLEMMultiTaskLoss` (unchanged)

Same `SmallUNet` backbone and `PLEMMultiTaskLoss` configuration as `train_unet_joint.ipynb` -- the
per-source class-masking mechanism already generalizes to a third source with zero code change
here (see `datasets/joint.py::SOURCE_CLASSES`).


In [7]:
model = SmallUNet(in_ch=3, num_classes=NUM_CLASSES, base=16).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"SmallUNet (4-class) parameter count: {n_params:,}")

CLASS_CONFIG = {
    1: {"name": "road", "tolerance": 10},
    2: {"name": "building", "tolerance": 2},
    3: {"name": "point", "tolerance": 3},
}

# Point-term weight -- history of two real-run failures on this term:
#   Run 1: model predicted ZERO point pixels on every source (heatmap sub-term
#     collapsed 1.13 -> 0.0004 after epoch 1). Cause: gt_centroids_to_heatmap
#     never produced a pixel == 1.0 for a sub-pixel (real multi-pixel-blob)
#     centroid, so heatmap_focal_loss's positive term (gated on
#     `gt_heatmap >= 1.0`) never fired. Fixed by stamping each centroid's
#     nearest integer pixel to 1.0 (losses/heatmap.py).
#   Run 2 (that fix + POINT_LOSS_WEIGHT=2.0 + 6x point-patch oversampling all
#     combined for the first time): heatmap sub-term spiked to 58.59
#     (unweighted) in epoch 1 -- with weight=2.0, ~97% of that epoch's total
#     backprop signal (train_loss=120.35, of which ce_dice+tolerance+cldice
#     combined only ~3.16) -- and the shared encoder/decoder never recovered:
#     tolerance/cldice stayed essentially flat for the whole 27-epoch run
#     (early-stopped at a worse val_loss, 2.3363 vs. run 1's 1.5067), and the
#     BUILDING channel (supervised by an unrelated loss term, sharing only the
#     trunk) collapsed to 0 predicted pixels on every source as collateral
#     damage. Point predictions were STILL zero despite the run-1 fix. Cause:
#     heatmap_focal_loss normalized its negative term (summed over up to
#     B*H*W ~= 3M full-resolution pixels) by the same tiny `num_pos` count as
#     the positive term -- a batch with few peak pixels produced a wildly
#     oversized loss. Fixed in losses/heatmap.py by normalizing the negative
#     term by its own (always-large) pixel count instead of `num_pos`,
#     decoupling the two. With that fix, the positive term's per-instance
#     signal is no longer diluted (it never needed the 2x weight to
#     compensate -- that was a misdiagnosis the first time), so
#     POINT_LOSS_WEIGHT is reverted to a neutral 1.0. Also added: gradient
#     clipping in the training loop below, as a second line of defense against
#     any single batch's loss spike (from this or any other term) destabilizing
#     the shared trunk the way run 2's did. Re-check the heatmap sub-term vs.
#     ce_dice in epoch 1 of the next run: it should be the same order of
#     magnitude, not ~30x larger.
POINT_LOSS_WEIGHT = 1.0

loss_fn = PLEMMultiTaskLoss(
    class_config=CLASS_CONFIG, linear_classes=[1], polygon_classes=[2], point_classes=[3],
    weights={"heatmap": POINT_LOSS_WEIGHT},
)


SmallUNet (4-class) parameter count: 483,492


## Training loop -- GPU-scaled, multi-epoch, mixed precision, periodic checkpointing, early stopping

Unlike the 2-epoch CPU proof of concept, this is a real multi-epoch GPU run long enough to be worth
interrupting/resuming, so it gets: automatic mixed precision (`torch.cuda.amp`), a cosine LR
schedule, a periodic rolling checkpoint every `CHECKPOINT_EVERY` epochs (in addition to the
existing best-val-loss checkpoint) so an interrupted run doesn't lose all progress, and **early
stopping** on validation loss (`PATIENCE = 3` epochs, deliberately tight/sensitive rather than the
more common 5-10 -- this run's `EPOCHS=40` budget is a ceiling, not a target, and stopping promptly
once val_loss stops improving is preferred over burning cloud-GPU hours on a plateaued run). Note
the `CosineAnnealingLR` schedule is still built for the full `T_max=EPOCHS`, so an early-stopped run
ends before the LR has fully annealed to its floor -- an accepted trade-off, not a bug.


In [ ]:
EPOCHS = 40
LR = 2e-3
CHECKPOINT_EVERY = 5
PATIENCE = 3  # sensitive: stop after 3 epochs with no val_loss improvement
MIN_DELTA = 1e-4  # improvement smaller than this doesn't reset the patience counter
GRAD_CLIP_NORM = 5.0  # added after run 2's epoch-1 loss spike (see loss-config cell) destabilized
                        # the shared trunk with no clipping in place; bounds any single batch's
                        # update regardless of which loss term produced an outsized gradient.
BEST_CHECKPOINT_PATH = MODELS_DIR / "unet_joint_4class_scaled.pt"
LATEST_CHECKPOINT_PATH = MODELS_DIR / "unet_joint_4class_scaled_latest.pt"

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

history = {"epoch": [], "train_loss": [], "val_loss": []}
sub_term_history = {"epoch": []}
best_val_loss = float("inf")
epochs_no_improve = 0
stopped_early = False

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_n = 0.0, 0
    sub_sums = {}
    for images, labels, class_masks in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        class_masks = class_masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out = loss_fn(model(images), labels, class_masks)
        scaler.scale(out["loss"]).backward()
        # Unscale before clipping so max_norm is measured in true gradient units,
        # not the AMP loss-scale-inflated ones.
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        bs = images.size(0)
        train_loss_sum += out["loss"].item() * bs
        train_n += bs
        for k, v in out.items():
            if k == "loss":
                continue
            sub_sums[k] = sub_sums.get(k, 0.0) + v * bs
    scheduler.step()
    train_loss = train_loss_sum / train_n

    model.eval()
    val_loss_sum, val_n = 0.0, 0
    with torch.no_grad():
        for images, labels, class_masks in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            class_masks = class_masks.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast():
                out = loss_fn(model(images), labels, class_masks)
            val_loss_sum += out["loss"].item() * images.size(0)
            val_n += images.size(0)
    val_loss = val_loss_sum / val_n

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    sub_term_history["epoch"].append(epoch)
    for k, total in sub_sums.items():
        sub_term_history.setdefault(k, []).append(total / train_n)

    sub_str = "  ".join(f"{k}={total / train_n:.4f}" for k, total in sub_sums.items())
    print(f"epoch {epoch:3d}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"lr={scheduler.get_last_lr()[0]:.2e}  [{sub_str}]")

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_CHECKPOINT_PATH)
        print(f"  -> new best val_loss, checkpoint saved to {BEST_CHECKPOINT_PATH}")
    else:
        epochs_no_improve += 1
        print(f"  -> no improvement for {epochs_no_improve}/{PATIENCE} epoch(s) "
              f"(best_val_loss={best_val_loss:.4f})")

    if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS:
        torch.save({
            "epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
        }, LATEST_CHECKPOINT_PATH)
        print(f"  -> rolling checkpoint saved to {LATEST_CHECKPOINT_PATH} (resumable)")

    if epochs_no_improve >= PATIENCE:
        stopped_early = True
        print(f"\nEarly stopping: val_loss hasn't improved by >= {MIN_DELTA} for "
              f"{PATIENCE} consecutive epochs (stopped after epoch {epoch}/{EPOCHS}).")
        break

model.load_state_dict(torch.load(BEST_CHECKPOINT_PATH, map_location=DEVICE))
model.eval()
status = "stopped early" if stopped_early else "ran to EPOCHS"
print(f"\nBest val_loss={best_val_loss:.4f} ({status}); loaded that checkpoint for evaluation.")

sub_term_df = pd.DataFrame(sub_term_history)
print("\nPer-epoch sub-term train loss (verification: are ALL terms decreasing, not just the total?):")
display(sub_term_df)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["epoch"], history["train_loss"], color="#2a78d6", label="train_loss")
ax.plot(history["epoch"], history["val_loss"], color="#eb6834", label="val_loss")
ax.set_xlabel("epoch")
ax.set_ylabel("PLEMMultiTaskLoss (total)")
ax.set_title("Joint 4-class scaled training curve")
ax.legend()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for k in ["ce_dice", "tolerance", "cldice", "heatmap"]:
    if k in sub_term_df.columns:
        ax.plot(sub_term_df["epoch"], sub_term_df[k], label=k)
ax.set_xlabel("epoch")
ax.set_ylabel("train loss (sub-term)")
ax.set_title("Per-term training loss -- each should trend down, not just the total")
ax.legend()
fig.tight_layout()
plt.show()


## Test-set evaluation with PLEM's metrics (normal brightness)

Same pattern as `train_unet_joint.ipynb`, now with a 3-valued `source` column and the
**current-iteration composites as the headline metrics**: `cbhm_soft` (pixel-area-weighted
arithmetic mean -- no zero-collapse when one sparse class fails) and `dtaf1_topo`
(F1 blended with raster-skeleton APLS for linear classes -- closes DTAF1's road-breakage blind
spot). `evaluate_all()` computes both alongside the older harsh `cbhm`/`dtaf1`, which are kept in
the table only for comparison. `dtaf1_topo` differs from `dtaf1` only for SpaceNet (the sole
source with a linear class); this is also the first run of DTAF1-Topo/APLS on real data.


In [ ]:
import hashlib

from metrics import point_f1


def stable_seed(s):
    """Process-stable 32-bit seed from a string. Python's built-in hash() is
    salted per process, so `abs(hash(x)) % 2**32` is not reproducible run to run
    despite the "fixed severities" claim in the dark-test section."""
    return int(hashlib.sha256(str(s).encode()).hexdigest()[:8], 16)


# Headline metrics are the *current* iterations: CBHM-soft (pixel-area-weighted
# arithmetic mean -- doesn't zero-collapse when one sparse class fails) and
# DTAF1-Topo (blends per-class F1 with raster-skeleton APLS for linear classes,
# closing DTAF1's road-breakage blind spot). Both are computed by evaluate_all()
# alongside the older cbhm/dtaf1, which are kept in the table only as the harsh
# foils. NOTE: DTAF1-Topo only differs from DTAF1 for SpaceNet tiles (the only
# source with a linear class); for Potsdam/SpaceNet6 (linear_classes=[]) APLS is
# skipped and dtaf1_topo == dtaf1. This is also the first time DTAF1-Topo/APLS
# runs on real data -- previously synthetic-only (see CLAUDE.md).
#
# Score each tile ONLY on the feature types its own source annotates. Passing the
# point class in a shared dtaf1_config (or linear_classes=[1] to a road-less
# source) drags DTAF1/CBHM to ~0 for Potsdam/SpaceNet6 regardless of prediction
# quality -- see metrics/unified.py::evaluate_all. cbhm()/dtaf1_topo() degenerate
# to the building score when linear_classes is empty instead of collapsing to 0.
_SOURCE_EVAL = {
    "spacenet":  dict(linear_classes=[1], polygon_classes=[2], point_classes=None,
                      dtaf1_config={k: CLASS_CONFIG[k] for k in (1, 2)}),
    "potsdam":   dict(linear_classes=[], polygon_classes=[2], point_classes=[3],
                      dtaf1_config={k: CLASS_CONFIG[k] for k in (2, 3)}),
    "spacenet6": dict(linear_classes=[], polygon_classes=[2], point_classes=None,
                      dtaf1_config={k: CLASS_CONFIG[k] for k in (2,)}),
}

# cbhm_soft / dtaf1_topo lead; cbhm / dtaf1 kept alongside for the harsh-foil view.
_METRIC_COLS = ["cbhm_soft", "dtaf1_topo", "dtaf1_topo_weighted",
                "cbhm", "dtaf1", "dtaf1_weighted",
                "cldice_mean", "bf_mean", "point_f1_mean"]


def evaluate_tile(pred, gt, source):
    # point_min_area=2 drops single-pixel speckle in the point channel (a dark/SAR
    # prediction can scatter 10^4+ 1-px blobs and stall the dark-test sweep).
    return evaluate_all(pred, gt, point_min_area=2, **_SOURCE_EVAL[source])


test_rows = []
test_preds = {}

for t in test_tiles:
    pred = predict_tile(model, t["image"], t["label"].shape)
    test_preds[t["tile"]] = pred
    result = evaluate_tile(pred, t["label"], t["source"])
    row = {
        "source": t["source"], "tile": t["tile"],
        **{k: result[k] for k in _METRIC_COLS},
        "n_gt_points": 0, "n_pred_points": 0,
    }
    # point_f1_mean above counts a tile with no GT points AND no predicted points
    # as a perfect 1.0 (see metrics/point_f1.py) -- so the aggregate is inflated
    # by every point-free Potsdam crop. Record the raw instance counts so the
    # honest point score (Potsdam tiles that actually contain points) can be
    # separated out below.
    if t["source"] == "potsdam":
        pf = point_f1(pred == 3, t["label"] == 3, tolerance=5, min_area=2)
        row["n_gt_points"] = pf["n_gt"]
        row["n_pred_points"] = pf["n_pred"]
    test_rows.append(row)

test_results = pd.DataFrame(test_rows)
for col in _METRIC_COLS:
    test_results[col] = pd.to_numeric(test_results[col], errors="coerce")
test_results.to_csv(DATA_DIR / "train_unet_joint_scaled_test_results.csv", index=False)

# Headline (current-iteration) metrics per source, then the harsh foils beneath.
print("Current-iteration headline metrics (CBHM-soft / DTAF1-Topo):")
print(test_results.groupby("source")[
    ["cbhm_soft", "dtaf1_topo", "bf_mean", "point_f1_mean"]
].mean())
print("\nHarsh foils (older CBHM / DTAF1) for comparison:")
print(test_results.groupby("source")[["cbhm", "dtaf1"]].mean())

# --- Honest point-feature summary -------------------------------------------
pot = test_results[test_results["source"] == "potsdam"]
pot_pts = pot[pot["n_gt_points"] > 0]
n_pot, n_pot_pts = len(pot), len(pot_pts)
total_pred_pts = int(pot["n_pred_points"].sum())
print("\n--- Point-feature reality check (Potsdam test tiles) ---")
print(f"  tiles: {n_pot} total, {n_pot_pts} contain >=1 GT point instance")
print(f"  predicted point instances across ALL Potsdam test tiles: {total_pred_pts}")
print(f"  point_f1_mean incl. point-free tiles (inflated by trivial 1.0s): "
      f"{pot['point_f1_mean'].mean():.3f}")
if n_pot_pts:
    print(f"  point_f1_mean on the {n_pot_pts} tiles WITH GT points (honest): "
          f"{pot_pts['point_f1_mean'].mean():.3f}")
else:
    print("  (no Potsdam test tile has GT points -- honest point score undefined)")
if total_pred_pts == 0:
    print("  WARNING: model predicted ZERO point pixels anywhere -- point head has "
          "collapsed. Check the heatmap sub-term, POINT_LOSS_WEIGHT, POINT_OVERSAMPLE.")

test_results


In [ ]:

# Verification: training is per-source masked (PLEMMultiTaskLoss only supervises the
# classes each tile's source actually annotates -- see SOURCE_CLASSES/class_mask above),
# but prediction is NOT source-conditioned: predict_tile() always runs a plain unmasked
# argmax over all 4 channels regardless of t["source"]. (evaluate_tile() above IS scored
# per-source -- only on the classes each source annotates -- but that's a metric-reporting
# choice, not something predict_tile sees.) This block makes the inference guarantee
# explicit rather than implicit in the code path.
import inspect

sig = inspect.signature(predict_tile)
assert "class_mask" not in sig.parameters and "source" not in sig.parameters, (
    "predict_tile() must not accept a class_mask/source argument -- predictions are required "
    "to span all 3 non-background dimensions (road/building/point) on every test tile "
    "regardless of which classes that tile's own source provides ground truth for. If this "
    "assert fires, someone added source-conditioned masking to inference and broken that "
    "guarantee."
)

class_pixel_counts = {}  # source -> {class_id: predicted pixel count}
for t in test_tiles:
    counts = class_pixel_counts.setdefault(t["source"], {1: 0, 2: 0, 3: 0})
    pred = test_preds[t["tile"]]
    for c in (1, 2, 3):
        counts[c] += int((pred == c).sum())

pred_pixel_counts = pd.DataFrame([
    {"source": source, **{CLASS_NAMES[c]: n for c, n in counts.items()}}
    for source, counts in sorted(class_pixel_counts.items())
])
print("Predicted pixel counts per class, per test-tile source:")
display(pred_pixel_counts)

print("\nClasses each source's own ground truth never annotates, and whether the model "
      "predicted them anyway (evidence of full-dimension, GT-independent prediction):")
for source in sorted(by_source):
    annotated = set(SOURCE_CLASSES[source])
    unannotated = {1, 2, 3} - annotated
    if not unannotated:
        print(f"  {source}: annotates all 3 classes -- nothing to check.")
        continue
    row = pred_pixel_counts.loc[pred_pixel_counts["source"] == source].iloc[0]
    for c in sorted(unannotated):
        px = int(row[CLASS_NAMES[c]])
        print(f"  {source}: predicted {CLASS_NAMES[c]!r} on {px} px "
              f"(a class this source's ground truth never contains).")


## Dark-test evaluation: does the model actually get more robust to dark/night input?

This is the headline evidence for the night-robustness claim, not just "dark examples were
included in training." Each real test tile's image is darkened via `simulate_low_light` at fixed,
reproducible severities (`0.0` = untouched baseline), then run through the same stitched-inference
+ `evaluate_all()` pipeline as the normal-brightness evaluation above. Reported on the
current-iteration composites (`cbhm_soft`, `dtaf1_topo`) plus `bf_mean` and the honest
`point_f1`. A model that's actually learned to be robust should degrade gracefully as severity
increases, not collapse the way an untrained-for-darkness model would.


In [ ]:
DARK_TEST_SEVERITIES = [0.0, 0.3, 0.6, 0.9]
# Current-iteration headline metrics: CBHM-soft + DTAF1-Topo (see the test-eval cell).
DARK_PLOT_METRICS = ["cbhm_soft", "dtaf1_topo", "bf_mean", "point_f1_mean"]
_METRIC_COLORS = {"cbhm_soft": "#2a78d6", "dtaf1_topo": "#eb6834",
                  "bf_mean": "#2ca25f", "point_f1_mean": "#8856a7"}
_DARK_COLS = ["cbhm_soft", "dtaf1_topo", "cbhm", "dtaf1", "cldice_mean", "bf_mean", "point_f1_mean"]
dark_rows = []

for severity in DARK_TEST_SEVERITIES:
    for t in test_tiles:
        if severity == 0.0:
            img = t["image"]
        else:
            img = simulate_low_light(
                t["image"], np.random.default_rng(stable_seed(t["tile"])), severity=severity,
            )
        pred = predict_tile(model, img, t["label"].shape)
        result = evaluate_tile(pred, t["label"], t["source"])
        n_gt_pts = n_pred_pts = 0
        if t["source"] == "potsdam":
            pf = point_f1(pred == 3, t["label"] == 3, tolerance=5, min_area=2)
            n_gt_pts, n_pred_pts = pf["n_gt"], pf["n_pred"]
        dark_rows.append({
            "source": t["source"], "tile": t["tile"], "severity": severity,
            **{k: result[k] for k in _DARK_COLS},
            "n_gt_points": n_gt_pts, "n_pred_points": n_pred_pts,
        })

dark_results = pd.DataFrame(dark_rows)
for col in _DARK_COLS:
    dark_results[col] = pd.to_numeric(dark_results[col], errors="coerce")
dark_results.to_csv(DATA_DIR / "train_unet_joint_scaled_dark_test_results.csv", index=False)

dark_summary = dark_results.groupby(["source", "severity"])[DARK_PLOT_METRICS].agg(["mean", "std"])
print("Mean +/- std, per (source, severity) -- higher severity = darker input.")
print("cbhm_soft reduces to the building score for potsdam/spacenet6; dtaf1_topo == dtaf1 there "
      "(no linear class); point_f1_mean is potsdam-only.")
display(dark_summary)

# Honest point score vs. darkness: only Potsdam tiles that actually contain GT
# points (point-free tiles score a trivial 1.0 and would mask any real change).
_pt = dark_results[(dark_results["source"] == "potsdam") & (dark_results["n_gt_points"] > 0)]
if len(_pt):
    print("\nHonest point_f1 (Potsdam tiles with GT points) + predicted point instances, by severity:")
    print(_pt.groupby("severity").agg(
        point_f1_honest=("point_f1_mean", "mean"),
        pred_point_instances=("n_pred_points", "sum"),
        n_tiles=("tile", "nunique"),
    ))
else:
    print("\n(no Potsdam test tile has GT points -- honest dark-point trend undefined)")

fig, axes = plt.subplots(1, len(by_source), figsize=(5.5 * len(by_source), 4), sharey=True)
if len(by_source) == 1:
    axes = [axes]
for ax, source in zip(axes, sorted(by_source)):
    sub = dark_results[dark_results["source"] == source]
    means = sub.groupby("severity")[DARK_PLOT_METRICS].mean()
    stds = sub.groupby("severity")[DARK_PLOT_METRICS].std()
    for metric in DARK_PLOT_METRICS:
        if means[metric].notna().any():
            ax.plot(means.index, means[metric], marker="o",
                    color=_METRIC_COLORS[metric], label=metric)
            ax.fill_between(means.index, means[metric] - stds[metric].fillna(0),
                             means[metric] + stds[metric].fillna(0),
                             color=_METRIC_COLORS[metric], alpha=0.15)
    ax.set_title(source)
    ax.set_xlabel("dark-simulation severity")
    ax.legend()
axes[0].set_ylabel("metric value")
fig.suptitle("Metric vs. darkening severity, per source (mean +/- std band)")
fig.tight_layout()
plt.show()


## Qualitative review: image / GT / prediction

Four blocks of 3-panel (GT overlay / prediction overlay / raw mask) figures:

1. **Score spread** -- worst / median / best test tile by `cbhm_soft`, all sources pooled.
2. **Best-performing tiles per source** -- top 2 by `cbhm_soft` for each of SpaceNet / Potsdam /
   SpaceNet6, i.e. what the model looks like when it works, per modality.
3. **Point-feature spotlight** -- the Potsdam test tiles that actually contain GT point
   instances, ranked by `point_f1` (point-free tiles score a trivial 1.0 and are excluded).
   This is the honest visual check on whether the point head produces anything, after the
   `losses/heatmap.py` peak-stamp fix + `POINT_LOSS_WEIGHT`/`POINT_OVERSAMPLE` changes.
4. **Darkened best-case tile** -- the top-`cbhm_soft` tile re-run at severity 0.6, a direct visual
   companion to the dark-test table.


In [ ]:
tiles_by_stem = {t["tile"]: t for t in test_tiles}


def show_examples(stems, suptitle):
    """3-panel (GT overlay / prediction overlay / raw mask) row per tile."""
    stems = [s for s in stems if s in tiles_by_stem]
    if not stems:
        print(f"(no tiles to show for: {suptitle})")
        return
    fig, axes = plt.subplots(len(stems), 3, figsize=(13.5, 4.4 * len(stems)))
    if len(stems) == 1:
        axes = axes[None, :]
    for row, stem in enumerate(stems):
        t = tiles_by_stem[stem]
        gt, pred, image = t["label"], test_preds[stem], t["image"]
        s = test_results.loc[test_results["tile"] == stem].iloc[0]
        pf1 = s["point_f1_mean"]
        pf1_str = f"{pf1:.2f}" if pd.notna(pf1) else "n/a"
        axes[row, 0].imshow(image)
        axes[row, 0].imshow(overlay4(gt), alpha=0.5)
        axes[row, 0].set_title(f"{stem} ({t['source']}) -- GT")
        axes[row, 1].imshow(image)
        axes[row, 1].imshow(overlay4(pred), alpha=0.5)
        axes[row, 1].set_title(f"pred -- cbhm_soft={s['cbhm_soft']:.2f} "
                               f"dtaf1_topo={s['dtaf1_topo']:.2f} "
                               f"bf={s['bf_mean']:.2f} pointF1={pf1_str}")
        axes[row, 2].imshow(overlay4(pred))
        axes[row, 2].set_title("raw prediction mask")
        for col in range(3):
            axes[row, col].axis("off")
    fig.suptitle(suptitle, fontsize=13)
    fig.tight_layout()
    plt.show()


# 1. Full score spread: worst / median / best by CBHM-soft across all sources.
order = test_results.sort_values("cbhm_soft")["tile"].tolist()
spread = [order[i] for i in np.linspace(0, len(order) - 1, 3).round().astype(int)]
show_examples(spread, "Score spread by CBHM-soft: worst -> median -> best (all sources)")

# 2. Best-performing tiles per source (top 2 by CBHM-soft each) -- what the model
#    looks like when it works, per modality.
best_stems = []
for src in sorted(by_source):
    best_stems += (test_results[test_results["source"] == src]
                   .sort_values("cbhm_soft", ascending=False)["tile"].head(2).tolist())
show_examples(best_stems, "Best-performing test tiles per source (top 2 by CBHM-soft)")

# 3. Point-feature spotlight: Potsdam tiles that ACTUALLY contain GT points,
#    ranked by point_f1 -- the honest visual on whether the point head works
#    at all (point-free tiles score a trivial 1.0 and tell us nothing).
pt_spot = (test_results[(test_results["source"] == "potsdam")
                        & (test_results["n_gt_points"] > 0)]
           .sort_values("point_f1_mean", ascending=False))
if len(pt_spot):
    show_examples(pt_spot["tile"].head(3).tolist(),
                  "Point-feature spotlight: best point_f1 among Potsdam tiles with GT points")
    print(pt_spot[["tile", "n_gt_points", "n_pred_points", "point_f1_mean", "cbhm_soft"]]
          .to_string(index=False))
else:
    print("No Potsdam test tile contains GT point instances -- point spotlight skipped.")

# 4. Darkened example: take the best-CBHM-soft tile and re-run at severity 0.6
#    (matching the dark-test sweep) -- does darkening break a good prediction?
dark_stem = best_stems[0] if best_stems else spread[-1]
t = tiles_by_stem[dark_stem]
dark_image = simulate_low_light(
    t["image"], np.random.default_rng(stable_seed(dark_stem)), severity=0.6,
)
dark_pred = predict_tile(model, dark_image, t["label"].shape)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
axes[0].imshow(dark_image)
axes[0].imshow(overlay4(t["label"]), alpha=0.5)
axes[0].set_title(f"{dark_stem} (darkened, severity=0.6) -- GT")
axes[1].imshow(dark_image)
axes[1].imshow(overlay4(dark_pred), alpha=0.5)
axes[1].set_title(f"{dark_stem} (darkened) -- prediction")
axes[2].imshow(overlay4(dark_pred))
axes[2].set_title("raw prediction mask (darkened input)")
for col in range(3):
    axes[col].axis("off")
fig.suptitle("Darkening a best-case tile: robustness spot-check", fontsize=13)
fig.tight_layout()
plt.show()


## Observations

- First joint 0D/1D/2D training run over **three** heterogeneous sources (SpaceNet: road+building;
  Potsdam: building+point; SpaceNet6: building only, real SAR) -- `SOURCE_CLASSES`'s per-source
  masking mechanism generalized to a third source with zero changes to `losses/multitask.py` or
  `models/unet.py`, confirming the design decision documented in `CLAUDE.md`.
- **Headline metrics are now `cbhm_soft` and `dtaf1_topo`** (the current iterations), not the
  original `cbhm`/`dtaf1`. `dtaf1_topo` is wired into `evaluate_all()` and blends per-class F1 with
  raster-skeleton APLS for linear classes -- so on SpaceNet a road that is positionally accurate
  but *fragmented* now scores lower than plain DTAF1 would report (DTAF1 stays pinned near 1.0
  under heavy road breakage -- its documented blind spot). This is the **first time DTAF1-Topo/APLS
  runs on real data**; APLS is deliberately harsh (skeleton-graph shortest-path ratios), so expect
  `dtaf1_topo` well below `dtaf1` on SpaceNet even for visually-decent predictions -- cross-check
  the `per_class_detail["apls"]` numbers and a few qualitative tiles before reading it as a
  regression. For Potsdam/SpaceNet6 (`linear_classes=[]`) `dtaf1_topo == dtaf1` exactly.
- **Point-head collapse: two runs, two distinct root causes, now both addressed.**
  - *Run 1* predicted ZERO point pixels on every source (heatmap sub-term collapsed
    1.13 -> 0.0004 after epoch 1). Cause: `gt_centroids_to_heatmap` never emitted a pixel `== 1.0`
    for a sub-pixel (real multi-pixel-blob) centroid, so `heatmap_focal_loss`'s positive term
    (gated on `gt_heatmap >= 1.0`) never fired. Fixed by stamping each centroid's nearest integer
    pixel to `1.0` (`losses/heatmap.py`).
  - *Run 2* (that fix + `POINT_LOSS_WEIGHT=2.0` + 6x point-patch oversampling, combined for the
    first time) **regressed overall model quality**: best val_loss 2.3363 vs. run 1's 1.5067,
    early-stopped after 27/40 epochs, and per-epoch sub-terms show `tolerance`/`cldice` essentially
    flat for the entire run. Point pixels were **still** zero, and -- new this run -- **the
    building channel also collapsed to 0 predicted pixels on every source** (spacenet/potsdam/
    spacenet6 alike), confirmed via the per-source predicted-pixel-count table. Root cause: the
    heatmap sub-term spiked to 58.59 (unweighted; ~117 with the 2x weight) in epoch 1 alone --
    ~97% of that epoch's total backprop signal (train_loss=120.35) -- because
    `heatmap_focal_loss` normalized its negative term (summed over up to B*H*W full-resolution
    pixels) by the same tiny `num_pos` peak-pixel count as the positive term. All four loss terms
    share one small (483K-param) trunk, so that one epoch's gradient was almost entirely dictated
    by the point term, plausibly knocking the shared representation into a bad basin that the
    (aggressively early-stopped) remaining epochs never escaped -- collateral damage to the
    building channel, not a building-specific bug.
  - **Fixes applied for the next run** (not yet re-executed -- this notebook's cell outputs above
    are from run 2 and predate these fixes): `losses/heatmap.py`'s negative term is now normalized
    by its own pixel count instead of `num_pos`, decoupling it from the (correctly undiluted)
    positive term; `POINT_LOSS_WEIGHT` reverted to `1.0` (the 2x weight was compensating for the
    wrong problem); gradient clipping (`max_norm=5.0`) added to the training loop as a second line
    of defense. Verify on the next run: (a) the heatmap sub-term should be the same order of
    magnitude as `ce_dice` in epoch 1, not ~30x larger; (b) `tolerance`/`cldice` should visibly
    decrease rather than stay flat; (c) the per-source predicted-pixel-count table should show
    nonzero building pixels on every source; (d) the "Point-feature reality check" printout
    (honest `point_f1` on Potsdam-with-points tiles + total predicted point-instance count) should
    show a nonzero count.
- **Night/dark-robustness evidence lives in the dark-test section above**, not in the normal-
  brightness test results -- that's the comparison that actually speaks to whether this run
  improved on the failure mode that motivated it (the proof-of-concept model failing on dark/night
  imagery). Given run 2's overall collapse, the dark-test numbers above should be re-read after the
  next run rather than treated as evidence either way.
- **Early stopping** (`PATIENCE = 3`, `MIN_DELTA = 1e-4`, deliberately tight) means `EPOCHS=40` is a
  ceiling, not a guaranteed epoch count -- check the training-loop cell's printed "stopped early" vs.
  "ran to EPOCHS" status before assuming any run trained the full budget. Because `CosineAnnealingLR`
  is still built for `T_max=EPOCHS`, an early-stopped run's LR schedule never reaches its floor --
  worth keeping in mind if a `PATIENCE=3` stop looks premature; a larger `PATIENCE` or a schedule
  keyed to actual stopped-epoch count is the fix if that turns out to matter in practice.
- **Known limitation, inherited and now spanning 3 sources**: SpaceNet, Potsdam, and SpaceNet6 are
  not resampled to a common GSD, and SpaceNet6's SAR imagery is additionally a different sensor
  modality entirely, compressed to pseudo-RGB via `sar_bands_to_pseudo_rgb`'s `"replicate"` mode --
  which discards SAR's phase/polarimetric structure that a dedicated encoder branch could have
  exploited. Accepted trade-off given this project's moderate scope; a real ablation of this choice
  (e.g. a small dedicated SAR encoder branch) is future work, not done here.
- **Comparison against `train_unet_joint.ipynb`'s 2-epoch/2-source run is directional only**:
  different epoch budget, different dataset scale/composition, and different random seed dynamics
  -- not an apples-to-apples ablation. The strongest follow-up evidence that the augmentation
  specifically (not just more data/epochs) helped would be a `DARK_AUG_PROB=0` re-run compared
  against this one on the same dark-test sweep -- noted as future work, not run here.
- If a `PLEMMultiTaskLoss` sub-term's per-epoch value stays flat near its starting value while the
  total still drops, that term isn't learning -- worth checking its `weights` entry before trusting
  its contribution to the final model (this is exactly what run 2's `tolerance`/`cldice` flatline
  looked like, and how it was caught).
